# Notebook 5 — Explainability with SHAP and LSTM Reconstruction Errors

**Goal:** When the model flags an anomaly, tell the operator *why* — which features are responsible.

**Two complementary approaches:**
1. **SHAP TreeExplainer** on Isolation Forest — attributes the anomaly score to each feature
2. **Per-feature reconstruction error** on LSTM Autoencoder — features that are hardest to reconstruct caused the anomaly

**Output:** for every flagged anomaly, a ranked list like:
```
Anomaly score: 0.87 (CRITICAL)
Top contributors:
  1. fan1: -38% (1500 RPM, baseline 2400)
  2. temp2_1: +18% (65°C, baseline 46°C)
  3. temp_max: +14% (62°C, baseline 47°C)
Diagnosis: fan failure causing thermal rise
```

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

sns.set_style('whitegrid')
DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'miner_telemetry_engineered.csv', parse_dates=['timestamp'])
with open(DATA_DIR / 'feature_sets.json') as f:
    feature_sets = json.load(f)
feats = [f for f in feature_sets['B_raw_domain'] if f in df.columns]
X = df[feats].fillna(0).values
y = (df['anomaly_label'] == 'anomaly').astype(int).values

split = int(len(df) * 0.7)
X_train_normal = X[:split][y[:split] == 0]
scaler = StandardScaler().fit(X_train_normal)
X_train_normal_s = scaler.transform(X_train_normal)
X_test_s = scaler.transform(X[split:])
model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
model.fit(X_train_normal_s)

## Method 1 — SHAP TreeExplainer on Isolation Forest

SHAP (SHapley Additive exPlanations) attributes the model's score to each input feature. For a flagged anomaly, the features with the largest SHAP magnitudes are the ones "responsible" for the alert.

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    # Pick 3 representative anomalies — one per type
    test_df = df.iloc[split:].reset_index(drop=True)
    anomaly_indices = {}
    for t in ['fan_failure', 'chip_degradation', 'voltage_instability']:
        cands = test_df.index[test_df['anomaly_type'] == t].tolist()
        if cands: anomaly_indices[t] = cands[len(cands)//2]

    for atype, idx in anomaly_indices.items():
        instance = X_test_s[idx:idx+1]
        shap_values = explainer.shap_values(instance)
        # SHAP returns array of shape (1, n_features) — flatten
        sv = np.array(shap_values).flatten()
        contribution = pd.DataFrame({
            'feature': feats,
            'shap_value': sv,
            'feature_value': X[split:][idx],
        })
        contribution['abs_shap'] = contribution['shap_value'].abs()
        contribution = contribution.sort_values('abs_shap', ascending=False).head(10)
        print(f'\n=== Anomaly type: {atype} ===')
        print(contribution[['feature','feature_value','shap_value']].round(4).to_string(index=False))

        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ['#ef4444' if v > 0 else '#3b82f6' for v in contribution['shap_value']]
        ax.barh(contribution['feature'], contribution['shap_value'], color=colors)
        ax.invert_yaxis(); ax.set_xlabel('SHAP value (anomaly contribution)')
        ax.set_title(f'Top features driving anomaly: {atype}')
        ax.axvline(0, color='black', linewidth=0.5)
        plt.tight_layout(); plt.show()
    SHAP_OK = True
except ImportError:
    print('SHAP not installed. Install: pip install shap')
    SHAP_OK = False
except Exception as e:
    print(f'SHAP failed: {e}')
    SHAP_OK = False

## Method 2 — Per-feature reconstruction error from LSTM Autoencoder

An LSTM autoencoder learns to reconstruct normal sequences. When given an anomalous sequence, it fails to reconstruct certain features more than others — those are the anomalous features.

In [ ]:
try:
    import torch
    import torch.nn as nn
    seq_len, hidden = 10, 32
    n_feat = X_train_normal_s.shape[1]
    Xs_tr = np.stack([X_train_normal_s[i:i+seq_len] for i in range(len(X_train_normal_s) - seq_len + 1)])
    class AE(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.LSTM(n_feat, hidden, batch_first=True)
            self.dec = nn.LSTM(hidden, n_feat, batch_first=True)
        def forward(self, x):
            h, _ = self.enc(x); o, _ = self.dec(h); return o
    ae = AE(); opt = torch.optim.Adam(ae.parameters(), lr=1e-3); lf = nn.MSELoss()
    Xt = torch.tensor(Xs_tr, dtype=torch.float32)
    print('Training LSTM autoencoder...')
    for epoch in range(15):
        idx = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 64):
            b = Xt[idx[i:i+64]]; opt.zero_grad()
            l = lf(ae(b), b); l.backward(); opt.step()
    print('Trained')

    def per_feature_recon_err(seq_arr):
        """Returns reconstruction error per feature (higher = more anomalous)."""
        with torch.no_grad():
            x = torch.tensor(seq_arr.reshape(1, seq_len, -1), dtype=torch.float32)
            recon = ae(x)
            err = ((x - recon) ** 2).mean(axis=1).flatten().numpy()
        return err

    test_df = df.iloc[split:].reset_index(drop=True)
    for atype in ['fan_failure', 'chip_degradation', 'voltage_instability']:
        idx_list = test_df.index[test_df['anomaly_type'] == atype].tolist()
        if not idx_list: continue
        idx = idx_list[len(idx_list)//2]
        if idx < seq_len: continue
        seq = X_test_s[idx-seq_len+1:idx+1]
        err = per_feature_recon_err(seq)
        ranked = sorted(zip(feats, err), key=lambda x: -x[1])[:10]
        print(f'\n=== {atype} — top 10 by per-feature reconstruction error ===')
        for f, e in ranked:
            actual = X[split:][idx, feats.index(f)]
            print(f'  {f:25} err={e:.4f}  current={actual:.2f}')
    LSTM_OK = True
except ImportError:
    print('PyTorch not installed — skip')
    LSTM_OK = False

## Combining both methods → diagnostic report

For each detected anomaly, compose a structured report combining SHAP and LSTM reconstruction error. This is what gets sent to the operator and (optionally) to the chatbot.

In [ ]:
def diagnose(idx, baseline_means, baseline_stds):
    """Generate a structured diagnostic report for sample idx in test set."""
    instance = X_test_s[idx:idx+1]
    raw = X[split:][idx]
    score = -model.decision_function(instance)[0]

    # SHAP contributions
    contrib = []
    if SHAP_OK:
        sv = explainer.shap_values(instance)
        sv = np.array(sv).flatten()
        for f, s, v in zip(feats, sv, raw):
            mean = baseline_means.get(f, 0)
            std  = baseline_stds.get(f, 0)
            pct = (v - mean) / abs(mean) * 100 if mean != 0 else 0
            contrib.append({
                'feature': f, 'value': round(float(v), 3),
                'baseline_mean': round(float(mean), 3),
                'pct_deviation': round(float(pct), 2),
                'shap': round(float(s), 4),
                'z_score': round(abs((v - mean) / (std + 1e-9)), 2),
            })
        contrib.sort(key=lambda x: -abs(x['shap']))
    return {'score': round(float(score), 4), 'top_contributors': contrib[:5]}

# Compute per-feature baseline from training normal samples
baseline_means = dict(zip(feats, X[:split][y[:split]==0].mean(axis=0)))
baseline_stds  = dict(zip(feats, X[:split][y[:split]==0].std(axis=0)))

# Diagnose 3 anomalies of different types
test_df = df.iloc[split:].reset_index(drop=True)
for atype in ['fan_failure', 'chip_degradation', 'voltage_instability']:
    idxs = test_df.index[test_df['anomaly_type'] == atype].tolist()
    if not idxs: continue
    idx = idxs[len(idxs)//2]
    rep = diagnose(idx, baseline_means, baseline_stds)
    print(f'\n=== Diagnosis: {atype} (sample {idx}, score {rep["score"]}) ===')
    for c in rep['top_contributors']:
        sign = '+' if c['pct_deviation'] > 0 else ''
        print(f'  {c["feature"]:25} = {c["value"]:8.2f} '
              f'(baseline {c["baseline_mean"]:7.2f}, dev {sign}{c["pct_deviation"]:.1f}%, '
              f'z={c["z_score"]}, shap={c["shap"]:+.3f})')

## Production integration

The `diagnose()` function above is what powers the **explainability** in the production system:
- The dashboard shows the top contributors as colored % deviations
- The chatbot receives the structured report as context
- The user sees "why" instead of just "what"

This file (`backend/ml/explainer.py`) implements the production version using only SHAP TreeExplainer (LSTM optional).